# Data Feasibility: California Wildfire Pilot

## Purpose

This notebook checks whether the proposed data sources can support the SmokeWatch Replay project.

The pilot focuses on the 2021 Dixie Fire in California during July 13–August 31, 2021. We will examine whether fire detections, air-quality observations, weather records, and incident information can be cleaned and joined into hourly county-level records.

## Questions

1. Can each source be accessed and downloaded?
2. What fields and time formats does each source contain?
3. Can records be aligned by county and hour?
4. What missing values, duplicates, and timestamp problems must be handled?
5. Is the proposed pilot feasible for the project timeline?

## Data Sources to Validate

| Source | Purpose in the pipeline | Initial validation |
|---|---|---|
| NASA FIRMS | Fire detection events | Check location, date, time, and fire-confidence fields |
| EPA AirData | Hourly PM2.5 observations | Check PM2.5 values, station location, and timestamps |
| NOAA | Weather observations | Check temperature, wind, and observation time |
| CAL FIRE | Fire incident information | Check incident name, dates, counties, and coordinates |

The sources will be evaluated for accessibility, time coverage, geographic coverage, missing values, duplicate records, and compatibility with an hourly county-level join.

## Source 1: NASA FIRMS Active Fire Data

- **Official archive:** https://firms.modaps.eosdis.nasa.gov/download/
- **Area API:** https://firms.modaps.eosdis.nasa.gov/api/area/
- **Role:** Provides satellite-detected active-fire locations and observation times.
- **Initial product:** VIIRS S-NPP 375 m Standard Processing (`VIIRS_SNPP_SP`)
- **Temporal fit:** VIIRS S-NPP coverage begins in 2012, so it includes the 2021 Dixie Fire period.
- **Available formats:** CSV, JSON, and Shapefile.
- **Expected fields:** Latitude, longitude, acquisition date, acquisition time, confidence, fire radiative power, satellite, and day/night indicator.
- **Access requirement:** Historical archive access requires an Earthdata login or email verification. Programmatic API access requires a free FIRMS map key.
- **Initial decision:** Begin with one VIIRS product to avoid duplicate detections from combining multiple satellites. Reconsider additional products only after validating the first sample.

## Source 2: EPA AirData Hourly PM2.5

- **Official download page:** https://aqs.epa.gov/aqsweb/airdata/download_files.html
- **2021 hourly file:** https://aqs.epa.gov/aqsweb/airdata/hourly_88101_2021.zip
- **Parameter:** PM2.5 FRM/FEM Mass (`88101`)
- **Role:** Provides hourly PM2.5 measurements from outdoor air-quality monitors.
- **Format:** Compressed ZIP containing a CSV file with a header row.
- **Dataset size:** Approximately 8.6 million national records and 67 MB compressed.
- **Important fields:** State Code, County Code, Site Num, latitude, longitude, Date Local, Time Local, Date GMT, Time GMT, Sample Measurement, units, qualifier, and method information.
- **Planned filtering:** Keep California records for Butte, Plumas, Shasta, Lassen, and Tehama counties during July 13–August 31, 2021.
- **Timestamp plan:** Use the GMT date and time fields to align EPA observations with NASA FIRMS timestamps.
- **Initial concern:** A county may have multiple monitoring sites or instruments during the same hour. We must define an aggregation rule, such as calculating the hourly county mean after quality checks.

## Source 3: NOAA Global Hourly Weather Data

- **Official product page:** https://www.ncei.noaa.gov/products/land-based-station/integrated-surface-database
- **Data search:** https://www.ncei.noaa.gov/access/search/datasets/global-hourly/
- **API documentation:** https://www.ncei.noaa.gov/access/search/documentation/data-service/
- **Role:** Provides hourly and sub-hourly weather observations from ground stations.
- **Expected variables:** Temperature, dew point, wind speed, wind direction, wind gust, atmospheric pressure, visibility, and precipitation.
- **Time coverage:** Historical data include the 2021 Dixie Fire period.
- **Planned filtering:** Select weather stations in or near Butte, Plumas, Shasta, Lassen, and Tehama counties for July 13–August 31, 2021.
- **Timestamp plan:** Convert observation timestamps to UTC and round or aggregate them into one-hour windows.
- **Initial concern:** NOAA records are organized by weather station rather than directly by county. Station coordinates must be mapped to counties, and some counties may not contain a suitable station.
- **Initial decision:** Validate station availability before selecting the final weather variables or downloading the complete pilot period.

## Source 4: CAL FIRE Dixie Fire Incident Information

- **Official 2021 incident archive:** https://www.fire.ca.gov/incidents/2021/
- **Official Dixie Fire page:** https://www.fire.ca.gov/incidents/2021/7/13/dixie-fire/
- **Role:** Provides authoritative incident context and metadata for the pilot case study.
- **Available information:** Incident name, location, counties, acreage, containment, status, cause, structures affected, and archived updates.
- **Geographic scope:** Butte, Plumas, Shasta, Lassen, and Tehama counties.
- **Final reported size:** 963,309 acres.
- **Planned use:** Use CAL FIRE information as an incident reference or dimension record rather than as the primary hourly event stream.
- **Initial concern:** CAL FIRE updates are not consistently hourly, and some archived pages show differences in recorded start dates or county coverage as the incident evolved.
- **Initial decision:** Use NASA, EPA, and NOAA for hourly observations. Use CAL FIRE to validate the selected incident, dates, counties, and final incident characteristics.

## Preliminary Feasibility Assessment

| Source | Preliminary status | Main risk | Decision |
|---|---|---|---|
| NASA FIRMS | Feasible | API requests are limited to five-day batches and require a map key | Proceed with one VIIRS product and automate date batches |
| EPA AirData | Feasible | National hourly file is large and may contain multiple monitors per county-hour | Filter by date and county, then define an aggregation rule |
| NOAA Global Hourly | Conditionally feasible | Weather stations may not be available inside every selected county | Validate station coverage before finalizing the geographic join |
| CAL FIRE | Feasible for metadata | Incident updates are not consistently hourly | Use as an incident reference rather than an hourly stream |

## Preliminary Conclusion

The proposed pilot is technically feasible because all four sources cover the 2021 period and provide downloadable historical information. NASA FIRMS, EPA AirData, and NOAA can support the time-based pipeline, while CAL FIRE can provide incident context. The most important unresolved issue is whether suitable NOAA and EPA monitoring stations exist in or near all five selected counties.